# DeepPavlov NER Fine-tuning: PERS / LOC / NORP

**Цель:** дообучить `DeepPavlov/rubert-base-cased-conversational` под 3 кастомных лейбла на небольшом пилоте.

**Pipeline:**
1. Загрузка
2. Фикс BIO-тегов
3. Tokenize + word-piece выравнивание лейблов
4. Обучение
5. Скачать модель

Зависимости

In [ ]:
!pip install -q transformers datasets seqeval openpyxl accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.9/250.9 kB 7.2 MB/s eta 0:00:00


Грузим датсет

In [ ]:
from google.colab import files
uploaded = files.upload()
XLSX_PATH = list(uploaded.keys())[0]
print('Загружен файл:', XLSX_PATH)

Saving Basic_NER.csv to Basic_NER (1).csv
Загружен файл: Basic_NER (1).csv


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(XLSX_PATH, encoding_errors='ignore')
print('Размер:', df.shape)
print('Колонки:', df.columns.tolist())
print('\nРаспределение лейблов:')
print(df['Label'].value_counts())

Размер: (615761, 5)
Колонки: ['Sentence_ID', 'Token', 'Label', 'Original_ID', 'Text']

Распределение лейблов:
Label
O         577963
B-LOC      13138
B-NORP     10453
B-PERS      9308
I-LOC       3346
I-PERS      1118
I-NORP       435
Name: count, dtype: int64


Немного фиксим данные

In [ ]:
labels = df['Label'].tolist()
fixed = labels.copy()

fixes = 0
for i in range(1, len(labels)):
    curr = labels[i]
    if not curr.startswith('I-'):
        continue
    # I-тег, но предыдущий — O
    if labels[i - 1] == 'O':
        entity_type = curr.split('-')[1]
        for j in range(i - 2, -1, -1):
            if labels[j] in (f'B-{entity_type}', f'I-{entity_type}'):
                for k in range(j + 1, i):
                    if labels[k] == 'O':
                        fixed[k] = f'I-{entity_type}'
                        fixes += 1
                break
            elif labels[j] not in ('O',):
                break

df['Label'] = fixed
print(f'Исправлено промежуточных O-тегов: {fixes}')
print('Новое распределение:')
print(pd.Series(fixed).value_counts())

Исправлено промежуточных O-тегов: 4726
Новое распределение:
O         573237
B-LOC      13138
B-NORP     10453
B-PERS      9308
I-LOC       7595
I-PERS      1118
I-NORP       912
Name: count, dtype: int64


Группируем токены в предложения

In [ ]:
sentences = []
for sent_id, group in df.groupby('Sentence_ID', sort=True):
    tokens = group['Token'].tolist()
    labels_list = group['Label'].tolist()
    sentences.append({'tokens': tokens, 'ner_tags': labels_list})

print(f'Всего предложений: {len(sentences)}')
print('\nПример #1:')
for tok, lbl in zip(sentences[0]['tokens'], sentences[0]['ner_tags']):
    if lbl != 'O':
        print(f'  {tok:25s} → {lbl}')

Всего предложений: 26842

Пример #1:
  Москвы                    → B-LOC


Label encoding

In [ ]:
LABEL_LIST = ['O', 'B-LOC', 'I-LOC', 'B-PERS', 'I-PERS', 'B-NORP', 'I-NORP']
label2id = {l: i for i, l in enumerate(LABEL_LIST)}
id2label = {i: l for l, i in label2id.items()}

print('Маппинг лейблов:', label2id)

Маппинг лейблов: {'O': 0, 'B-LOC': 1, 'I-LOC': 2, 'B-PERS': 3, 'I-PERS': 4, 'B-NORP': 5, 'I-NORP': 6}


Train / Val сплит

In [ ]:
import random
random.seed(42)

random.shuffle(sentences)
split = int(0.8 * len(sentences))
train_data = sentences[:split]
val_data   = sentences[split:]

print(f'Train: {len(train_data)} предложений, Val: {len(val_data)} предложений')

Train: 21473 предложений, Val: 5369 предложений


 Токенизация + выравнивание лейблов

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = 'DeepPavlov/rubert-base-cased-conversational'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_and_align(examples):
    """
    Для каждого слова BERT может нарезать несколько subword-токенов.
    Лейбл ставим только на первый subword, остальные -100 (игнорируются loss).
    """
    tokenized = tokenizer(
        examples['tokens'],
        truncation=True,
        is_split_into_words=True,
        padding='max_length',
        max_length=128
    )
    all_labels = []
    for i, label_seq in enumerate(examples['ner_tags']):
        word_ids = tokenized.word_ids(batch_index=i)
        prev_word = None
        label_ids = []
        for wid in word_ids:
            if wid is None:
                label_ids.append(-100)
            elif wid != prev_word:
                label_ids.append(label2id[label_seq[wid]])
            else:
                label_ids.append(-100)
            prev_word = wid
        all_labels.append(label_ids)
    tokenized['labels'] = all_labels
    return tokenized

# Оборачиваем в HuggingFace Dataset
from datasets import Dataset

def to_hf_dataset(data):
    return Dataset.from_dict({
        'tokens':   [d['tokens']   for d in data],
        'ner_tags': [d['ner_tags'] for d in data]
    })

train_ds = to_hf_dataset(train_data).map(tokenize_and_align, batched=True)
val_ds   = to_hf_dataset(val_data).map(tokenize_and_align, batched=True)

# Оставляем только нужные колонки
keep = ['input_ids', 'attention_mask', 'token_type_ids', 'labels']
train_ds = train_ds.remove_columns([c for c in train_ds.column_names if c not in keep])
val_ds   = val_ds.remove_columns([c for c in val_ds.column_names if c not in keep])

print('Train dataset:', train_ds)
print('Val dataset:',   val_ds)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/21473 [00:00<?, ? examples/s]

Map:   0%|          | 0/5369 [00:00<?, ? examples/s]

Train dataset: Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 21473
})
Val dataset: Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 5369
})


Загрузка модели

In [ ]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_LIST),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True   # classifier head пересоздаётся
)
print('Модель загружена. Параметров:', sum(p.numel() for p in model.parameters()))

pytorch_model.bin:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: DeepPavlov/rubert-base-cased-conversational
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight         

Модель загружена. Параметров: 177268231


In [ ]:
!pip install evaluate
!pip install seqeval

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.3 MB/s eta 0:00:00


Метрика seqeval

In [ ]:
import evaluate
seqeval = evaluate.load('seqeval')

def compute_metrics(p):
    preds, labels = p
    preds = np.argmax(preds, axis=2)

    true_preds, true_labels = [], []
    for pred_seq, label_seq in zip(preds, labels):
        tp, tl = [], []
        for p_id, l_id in zip(pred_seq, label_seq):
            if l_id != -100:
                tp.append(id2label[p_id])
                tl.append(id2label[l_id])
        true_preds.append(tp)
        true_labels.append(tl)

    result = seqeval.compute(predictions=true_preds, references=true_labels)
    return {
        'precision': result['overall_precision'],
        'recall':    result['overall_recall'],
        'f1':        result['overall_f1'],
        'accuracy':  result['overall_accuracy'],
    }

TrainingArguments

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir              = './ner_pilot',
    num_train_epochs        = 2,
    per_device_train_batch_size = 8,
    per_device_eval_batch_size  = 8,
    learning_rate           = 5e-5,
    weight_decay            = 0.01,
    warmup_ratio            = 0.1,
    eval_strategy           = 'epoch',
    save_strategy           = 'epoch',
    load_best_model_at_end  = True,
    metric_for_best_model   = 'f1',
    logging_steps           = 5,
    seed                    = 42,
    # fp16                    = True,      # если GPU
    bf16                    = True,
    report_to               = 'none',
    optim="adamw_torch",
)

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_ds,
    eval_dataset    = val_ds,
    compute_metrics = compute_metrics,
)

print('Trainer готов. Начинаем обучение...')

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Trainer готов. Начинаем обучение...


Обучение

In [ ]:
train_result = trainer.train()
print('\Обучение завершено!')
print(train_result.metrics)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.041016,0.041618,0.890836,0.904516,0.897624,0.988827
2,0.022852,0.040598,0.896112,0.907557,0.901798,0.989391


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


✅ Обучение завершено!
{'train_runtime': 1152.8448, 'train_samples_per_second': 37.264, 'train_steps_per_second': 4.658, 'total_flos': 2805539446516224.0, 'train_loss': 0.09705371821170856, 'epoch': 2.0}


Оценка на val

In [ ]:
eval_result = trainer.evaluate()
print('\nМетрики на val:')
for k, v in eval_result.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)



📊 Метрики на val:
  eval_loss: 0.0406
  eval_precision: 0.8961
  eval_recall: 0.9076
  eval_f1: 0.9018
  eval_accuracy: 0.9894
  eval_runtime: 15.5296
  eval_samples_per_second: 346.1770
  eval_steps_per_second: 43.2720
  epoch: 2.0000


Проверяем на примерах

In [ ]:
from transformers import pipeline

ner_pipe = pipeline(
    'ner',
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy='simple',   # склеивает subword-токены
    device=0                          # GPU
)

test_sentences = [
    'Мария Солопенко приехала из Москвы в Киев.',
    'Французы и немцы встретились в Берлине.',
    'Президент России Владимир Путин посетил Пекин.',
]

for sent in test_sentences:
    print(f'\n{sent}')
    for ent in ner_pipe(sent):
        print(f'   {ent["word"]:20s} → {ent["entity_group"]} ({ent["score"]:.2f})')


📝 Мария Солопенко приехала из Москвы в Киев.
   Москвы               → LOC (0.98)
   Киев                 → LOC (0.98)

📝 Французы и немцы встретились в Берлине.
   Французы             → NORP (0.96)
   немцы                → NORP (0.94)

📝 Президент России Владимир Путин посетил Пекин.
   России               → LOC (0.55)
   Пекин                → LOC (0.96)


Сохранение модели

In [1]:
SAVE_DIR = './my_ner_model'
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f'Модель сохранена в {SAVE_DIR}')

NameError: name 'model' is not defined

Скачиваем архив с моделью

In [ ]:
import shutil
from google.colab import files

archive_path = shutil.make_archive('my_ner_model', 'zip', SAVE_DIR)
print(f'Архив создан: {archive_path}')
files.download(archive_path)   # запустится скачивание в браузере

Архив создан: /content/my_ner_model.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>